In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# asyncpg: 비동기, httpx: 외부 API(http)비동기 호출
%pip install asyncpg httpx python-dotenv
%pip install requests


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("KAKAO_REST_API_KEY"))

73923fe6c9aff51e1c382ad85842e6ce


In [2]:
# 테스트용 초기 state 만들기
from core.state import make_initial_state
from core.mocks import mock_user_input

initial_state = make_initial_state(mock_user_input)

In [3]:
# [노드] preprocess_input
# 사용자 입력 전처리 (변수값 매핑 + 키워드 및 반경 설정)
from nodes.preprocess_input import preprocess_input

preprocess_result = preprocess_input(initial_state)
print("preprocess_input 결과:", preprocess_result)


preprocess_input 결과: {'user_input': {'destination': '홍대역', 'lat': 37.5572, 'lng': 126.9245, 'travel_days': 2, 'companion': 'couple', 'age_group': '20s', 'moods': ['active', 'healing', 'clean'], 'activities': ['thrill/experience', 'performance/culture', 'entertainment/sports', 'nature/walk'], 'transport': 'walk', 'avoid_activities': ['노래'], 'start_time': '09:00', 'end_time': '22:00', 'center_lat': 37.5572, 'center_lng': 126.9245, 'search_radius_km': 2.0, 'final_keywords': ['맛집', '카페', '스릴/체험', '공연/문화', '오락/스포츠', '자연/산책', '숙박'], 'companion_kr': '연인', 'age_group_kr': '20대', 'moods_kr': ['활기찬', '힐링', '깔끔한'], 'activities_kr': ['스릴/체험', '공연/문화', '오락/스포츠', '자연/산책'], 'transport_kr': '도보', 'duration_kr': '1박2일'}, 'warnings': [], 'step': 'preprocessed'}


In [4]:
# [노드] collect_candidate_pool
# kakao Local API로 raw 후보 풀 수집 + PostgreSQL
from nodes.collect_candidate_pool import collect_candidate_pool

candidates_result = await collect_candidate_pool(preprocess_result)
print("candidates_result 결과:", candidates_result)


candidates_result 결과: {'user_input': {'destination': '홍대역', 'lat': 37.5572, 'lng': 126.9245, 'travel_days': 2, 'companion': 'couple', 'age_group': '20s', 'moods': ['active', 'healing', 'clean'], 'activities': ['thrill/experience', 'performance/culture', 'entertainment/sports', 'nature/walk'], 'transport': 'walk', 'avoid_activities': ['노래'], 'start_time': '09:00', 'end_time': '22:00', 'center_lat': 37.5572, 'center_lng': 126.9245, 'search_radius_km': 2.0, 'final_keywords': ['맛집', '음식점', '카페', '찻집', '한옥카페', '루프탑카페', '방탈출카페', '만화카페', '동물카페', '짚라인', '번지점프', '놀이공원', '워터파크', '테마파크', 'VR체험', '이색체험', '서핑', '패러글라이딩', 'ATV', '공연장', '극장', '콘서트홀', '라이브하우스', '영화관', '뮤지컬', '오락실', '볼링장', '당구장', '사격장', '스포츠 시설', '클라이밍', 'PC방', '노래방', '보드게임카페', '공원', '산책로', '둘레길', '식물원', '해변', '바다', '해수욕장', '등산', '산', '계곡', '폭포', '갯벌', '숙박', '호텔', '게스트하우스', '펜션', '리조트', '모텔'], 'companion_kr': '연인', 'age_group_kr': '20대', 'moods_kr': ['활기찬', '힐링', '깔끔한'], 'activities_kr': ['스릴/체험', '공연/문화', '오락/스포츠', '자연/산책'], 'transpor

In [5]:
# [노드] first_filter_candidates
# 첫번쨰 필터 -> travel_days 기반 동적 cap으로 축약
from nodes.first_filter_candidates import first_filter_candidates

filter_result = first_filter_candidates(candidates_result, debug=True)

print(f"\n📌 warnings:")
for w in filter_result["warnings"]:
    print(f"   - {w}")



📦 시작
   ──────────────────────────────────────────────────
   ✂️  제거: 0개  |  남은: 854개
   📊 카테고리: :349  CE7:169  AD5:126  FD6:112  CT1:79  AT4:7
🔍 전체 장소 목록:
01. [FD6] 형님 저요 | 음식점 > 한식 > 육류,고기
02. [FD6] 비스트로주라 홍대점 | 음식점 > 양식
03. [FD6] 장인닭갈비 홍대점 | 음식점 > 한식 > 육류,고기 > 닭요리
04. [FD6] 동보성 | 음식점 > 중식 > 중국요리
05. [FD6] 마포곱창타운 | 음식점 > 한식 > 육류,고기 > 곱창,막창
06. [FD6] 츠케루 | 음식점 > 일식 > 일본식라면
07. [FD6] 오비야 | 음식점 > 일식 > 일식집
08. [FD6] 옛날집참숯구이 | 음식점 > 한식 > 육류,고기
09. [FD6] 미쓰족발 홍대본점 | 음식점 > 한식 > 육류,고기 > 족발,보쌈
10. [FD6] 하카타나카 | 음식점 > 일식
11. [CE7] 카페장쌤 | 음식점 > 카페
12. [CE7] 카페공명 연남점 | 음식점 > 카페 > 커피전문점
13. [FD6] 부부드꼼뜨와 | 음식점 > 양식 > 이탈리안
14. [FD6] 홍대최대포 | 음식점 > 한식 > 육류,고기
15. [FD6] 이츠모라멘 | 음식점 > 일식 > 일본식라면
16. [FD6] 육전국밥 서교점 | 음식점 > 한식 > 국밥
17. [CE7] 1984 | 음식점 > 카페 > 테마카페 > 북카페
18. [FD6] 꿀돼지집 | 음식점 > 한식 > 육류,고기 > 삼겹살
19. [FD6] 청기와갈비 | 음식점 > 한식 > 육류,고기 > 갈비
20. [FD6] 홍대조폭떡볶이 홍대2호점 | 음식점 > 분식 > 떡볶이
21. [CE7] 땡스네이쳐카페 | 음식점 > 카페
22. [FD6] 일편등심 | 음식점 > 한식 > 육류,고기
23. [FD6] 하이디라오 홍대지점 | 음식점 > 중식 > 중국요리
24. [FD6] 선셋클라

In [6]:
# [노드] second_filter_candidates
# 두번쨰 필터 -> 축소
from nodes.second_filter_candidates import second_filter_candidates

test_state = {
    "filtered_candidates": filter_result["filtered_candidates"],
    "user_input": filter_result["user_input"],
    "warnings": [],
}

second_filter_result = await second_filter_candidates(test_state)

print(f"⚠️ warnings: {second_filter_result['warnings']}")
print(f"✅ step: {second_filter_result['step']}")
print(f"📦 보강된 장소 수: {len(second_filter_result['filtered_candidates'])}")
print(f"📊 scored_candidates 수: {len(second_filter_result['scored_candidates'])}")
print(f"🏆 shortlist 수: {len(second_filter_result['shortlist'])}")

print("\n=== 전체 scored_candidates ===")

for i, s in enumerate(second_filter_result["scored_candidates"], 1):
    p = s["place"]

    print(f"""
{i}위. {p['name']}
  분위기: {p.get('atmosphere')}
  추천대상: {p.get('best_for')}
  활동: {p.get('place_tags')}
  재방문의사: {p.get('revisit_intent')}
  한줄요약: {p.get('summary')}
  mood_score: {s['mood_score']}
  party_fit_score: {s['party_fit_score']}
  total_score: {s['total_score']}
""")


print("\n=== shortlist ===")

for i, s in enumerate(second_filter_result["shortlist"], 1):
    p = s["place"]

    print(f"""
{i}위. {p['name']}
  total_score: {s['total_score']}
  bucket: {p.get('bucket')}
""")

⏱  네이버 블로그: 18.6초 (96개)
⏱  LLM 보강: 22.6초 (96개)
⚠️ warnings: []
✅ step: enriched
📦 보강된 장소 수: 96
📊 scored_candidates 수: 96
🏆 shortlist 수: 60

=== 전체 scored_candidates ===

1위. 홍스쭈꾸미 홍대본점
  분위기: ['활기찬', '깔끔한']
  추천대상: ['연인']
  활동: ['한식']
  재방문의사: high
  한줄요약: 매운 쭈꾸미 전문점
  mood_score: 66.7
  party_fit_score: 40
  total_score: 136.7


2위. 형님 저요
  분위기: ['활기찬']
  추천대상: ['연인', '친구']
  활동: ['고기']
  재방문의사: high
  한줄요약: 홍대 인기 고기 맛집
  mood_score: 33.3
  party_fit_score: 40
  total_score: 103.3


3위. 육전국밥 서교점
  분위기: ['힐링', '따뜻한']
  추천대상: ['연인']
  활동: ['한식']
  재방문의사: high
  한줄요약: 신선한 재료로 만든 국밥
  mood_score: 33.3
  party_fit_score: 40
  total_score: 103.3


4위. L7 홍대바이롯데
  분위기: ['힐링', '로맨틱']
  추천대상: ['연인']
  활동: ['숙소']
  재방문의사: high
  한줄요약: 편안한 홍대 롯데호텔 숙소
  mood_score: 33.3
  party_fit_score: 40
  total_score: 103.3


5위. 머큐어 앰배서더 서울 홍대
  분위기: ['깔끔한', '로맨틱']
  추천대상: ['연인']
  활동: ['숙소']
  재방문의사: high
  한줄요약: 홍대 근처의 앰배서더 호텔
  mood_score: 33.3
  party_fit_score: 40
  total_score: 103.3


6위. 아늑 호텔 홍대
  

In [7]:
# [노드] travel_matrix
# shortlist 30개 장소 간 이동시간 행렬 계산
from nodes.travel_matrix import travel_matrix

matrix_result = travel_matrix(second_filter_result)

# place_id → name 매핑
id_to_name = {item["place"]["id"]: item["place"]["name"] for item in second_filter_result["shortlist"]}

print(f"✅ place_index: {len(matrix_result['place_index'])}개")
print(f"✅ distance_matrix: {len(matrix_result['distance_matrix'])}x{len(matrix_result['distance_matrix'][0])}")
print(f"✅ time_matrix: {len(matrix_result['time_matrix'])}x{len(matrix_result['time_matrix'][0])}")

for i, from_id in enumerate(matrix_result['place_index']):
    print(f"\n📍 {id_to_name[from_id]} 에서:")
    for to_id, mins in zip(matrix_result['place_index'], matrix_result['time_matrix'][i]):
        if from_id == to_id:
            continue
        print(f"  → {id_to_name[to_id]}: {mins}분")

✅ place_index: 60개
✅ distance_matrix: 60x60
✅ time_matrix: 60x60

📍 홍스쭈꾸미 홍대본점 에서:
  → 형님 저요: 1.5분
  → 육전국밥 서교점: 7.6분
  → 지구별방탈출 홍대라스트시티점: 5.2분
  → 브레이크아웃이스케이프 홍대점: 5.7분
  → 경의선숲길: 4.5분
  → 경의선숲길: 9.9분
  → 리얼샷사격양궁장 홍대점: 9.7분
  → 액티브아레나 홍대점: 5.9분
  → 케이 탑건 홍대점: 2.5분
  → 홍제천: 23.1분
  → L7 홍대바이롯데: 6.1분
  → 머큐어 앰배서더 서울 홍대: 5.1분
  → 아늑 호텔 홍대: 7.6분
  → 아만티호텔서울 웨딩: 9.9분
  → 비스트로주라 홍대점: 4.1분
  → 장인닭갈비 홍대점: 4.8분
  → 동보성: 4.3분
  → 마포곱창타운: 6.7분
  → 츠케루: 5.2분
  → 이츠모라멘: 4.2분
  → 선셋클라우드: 2.4분
  → 벌툰 봉화당 홍대입구역점: 1.3분
  → 제로월드 홍대점: 10.2분
  → 호러스위치: 7.2분
  → 구름아래소극장: 4.6분
  → 소극장산울림: 5.6분
  → 미디어극장아이공: 2.5분
  → 윗잔다리어린이공원: 1.7분
  → 와우공원: 8.6분
  → 창천문화공원: 12.6분
  → 와우산: 8.7분
  → 노고산: 21.6분
  → 아로마인드: 19.1분
  → 마포아트센터 갤러리맥: 27.3분
  → 레이저아레나: 4.5분
  → 리얼샷사격양궁장: 9.8분
  → 호텔캠퍼스: 15.8분
  → 홀리데이 인 익스프레스 서울 홍대: 2.4분
  → 나비호스텔: 3.5분
  → 호스텔 클레오: 8.7분
  → Lazy cube: 5.7분
  → 소니픽쳐스엔터테인먼트코리아 극장배급지점: 18.9분
  → 니도나도모텔: 6.5분
  → 카페장쌤: 1.3분
  → 땡스네이쳐카페: 5.3분
  → 토라비: 1.3분
  → 디저트 머라이언: 2.2분
  → 이미커피: 5.9분
  → 오비야: 2.6

In [25]:
# [노드] plan_itinerary
import importlib
import utils.route.greedy_nn as gm
importlib.reload(gm)
import nodes.plan_itinerary as m
importlib.reload(m)
from nodes.plan_itinerary import plan_itinerary
from utils.route.route_check import check_route_intersections

plan_state = {**matrix_result, "shortlist": second_filter_result["shortlist"], "user_input": second_filter_result["user_input"]}
itinerary_result = plan_itinerary(plan_state)

print(f"전체 동선: {len(itinerary_result['all_routes'])}개")
print(f"이동시간 초과 제거: {itinerary_result['excluded_travel']}개")
print(f"bucket 연속 제거: {itinerary_result['excluded_bucket']}개")
print(f"place_tags 제거: {itinerary_result['excluded_tags']}개")
print(f"경로 교차 제거: {itinerary_result['excluded_cross']}개")
print(f"조건 통과: {len(itinerary_result['itineraries'])}개")

# 전체 동선 출력
for idx, itinerary in enumerate(itinerary_result["itineraries"], 1):
    print(f"\n{'='*50}")
    print(f"동선 {idx}번")
    print(f"{'='*50}")
    for item in itinerary:
        p = item["place"]
        tags = p.get("place_tags", [])
        print(f"{item['order']}. [{p.get('bucket','?'):8}] {p['name']}  {tags}")
        print(f"   도착: {item['arrive_at']}  출발: {item['leave_at']}  다음까지: {item['travel_to_next_minutes']}분")

    violations = check_route_intersections(itinerary)
    if violations:
        print("  ⚠️ 교차 발생:")
        for v in violations:
            print(f"    {v['detail']}")
    print()

전체 동선: 105개
이동시간 초과 제거: 10개
bucket 연속 제거: 0개
place_tags 제거: 56개
경로 교차 제거: 23개
조건 통과: 10개

동선 1번
1. [activity] 제로월드 홍대점  ['이색체험']
   도착: 09:00  출발: 11:00  다음까지: 6분
2. [food    ] 비스트로주라 홍대점  ['양식']
   도착: 11:06  출발: 12:36  다음까지: 4분
3. [cafe    ] 카페장쌤  ['카페']
   도착: 12:40  출발: 14:10  다음까지: 2분
4. [activity] 윗잔다리어린이공원  ['산책로']
   도착: 14:13  출발: 16:13  다음까지: 1분
5. [activity] 벌툰 봉화당 홍대입구역점  ['이색카페']
   도착: 16:15  출발: 18:15  다음까지: 0분
6. [food    ] 형님 저요  ['고기']
   도착: 18:15  출발: 19:45  다음까지: 5분
7. [lodging ] 니도나도모텔  ['숙소']
   도착: 19:50  출발: -  다음까지: 0분


동선 2번
1. [activity] 구름아래소극장  ['전시관/미술관']
   도착: 09:00  출발: 11:00  다음까지: 5분
2. [food    ] 이츠모라멘  ['일식']
   도착: 11:05  출발: 12:35  다음까지: 2분
3. [cafe    ] 토라비  ['카페']
   도착: 12:38  출발: 14:08  다음까지: 1분
4. [activity] 케이 탑건 홍대점  ['스포츠']
   도착: 14:09  출발: 16:09  다음까지: 3분
5. [activity] 윗잔다리어린이공원  ['산책로']
   도착: 16:13  출발: 18:13  다음까지: 1분
6. [food    ] 하이디라오 홍대지점  ['중식']
   도착: 18:15  출발: 19:45  다음까지: 4분
7. [lodging ] 머큐어 앰배서더 서울 홍대  ['숙소']
   도착: 19:49

In [19]:
# 경로 교차 발생 동선 확인
from utils.route.route_check import check_route_intersections

cross_count = 0
for r in itinerary_result["all_routes"]:
    violations = check_route_intersections(r["itinerary"])
    if violations:
        cross_count += 1
        names = [item["place"]["name"] for item in r["itinerary"]]
        print(f"\n동선: {' → '.join(names)}")
        for v in violations:
            print(f"  ⚠️ {v['detail']}")

print(f"\n총 교차 발생 동선: {cross_count}개")


동선: 땡스네이쳐카페 → 디저트 머라이언 → 소극장산울림 → 홍대조폭떡볶이 홍대2호점 → 커피빈 홍대역점 → 액티브아레나 홍대점 → 카페장쌤 → 하이디라오 홍대지점 → 레이저아레나 → L7 홍대바이롯데
  ⚠️ 땡스네이쳐카페 → 디저트 머라이언 와 레이저아레나 → L7 홍대바이롯데 교차
  ⚠️ 디저트 머라이언 → 소극장산울림 와 하이디라오 홍대지점 → 레이저아레나 교차
  ⚠️ 소극장산울림 → 홍대조폭떡볶이 홍대2호점 와 액티브아레나 홍대점 → 카페장쌤 교차
  ⚠️ 소극장산울림 → 홍대조폭떡볶이 홍대2호점 와 하이디라오 홍대지점 → 레이저아레나 교차
  ⚠️ 홍대조폭떡볶이 홍대2호점 → 커피빈 홍대역점 와 액티브아레나 홍대점 → 카페장쌤 교차
  ⚠️ 액티브아레나 홍대점 → 카페장쌤 와 하이디라오 홍대지점 → 레이저아레나 교차

동선: 토라비 → 벌툰 봉화당 홍대입구역점 → 홍대조폭떡볶이 홍대2호점 → 벌툰 인더스트리얼 홍대본점 → 리얼샷사격장 홍대점 → 커피빈 홍대역점 → 장인닭갈비 홍대점 → 디저트 머라이언 → 땡스네이쳐카페 → 머큐어 앰배서더 서울 홍대
  ⚠️ 홍대조폭떡볶이 홍대2호점 → 벌툰 인더스트리얼 홍대본점 와 리얼샷사격장 홍대점 → 커피빈 홍대역점 교차
  ⚠️ 홍대조폭떡볶이 홍대2호점 → 벌툰 인더스트리얼 홍대본점 와 장인닭갈비 홍대점 → 디저트 머라이언 교차
  ⚠️ 홍대조폭떡볶이 홍대2호점 → 벌툰 인더스트리얼 홍대본점 와 땡스네이쳐카페 → 머큐어 앰배서더 서울 홍대 교차
  ⚠️ 리얼샷사격장 홍대점 → 커피빈 홍대역점 와 장인닭갈비 홍대점 → 디저트 머라이언 교차
  ⚠️ 리얼샷사격장 홍대점 → 커피빈 홍대역점 와 땡스네이쳐카페 → 머큐어 앰배서더 서울 홍대 교차

동선: 지구별방탈출 홍대라스트시티점 → 디저트 머라이언 → 이츠모라멘 → 땡스네이쳐카페 → 리얼샷사격장 홍대점 → 케이 탑건 홍대점 → 형님 저요 → 브레이크아웃이스케이프 홍대점 → Lazy cube
  ⚠️ 지구별방탈출 홍대라스트시티점 → 디저트 머라이언 와 브레이크아

In [26]:
# [노드] select_itinerary
from nodes.select_itinerary import select_itinerary

select_state = {
    **itinerary_result,
    "user_input": second_filter_result["user_input"],
}
select_itinerary_result = await select_itinerary(select_state)

print(f"✅ step: {select_itinerary_result['step']}")
print(f"✅ 선택된 동선 day 수: {len(select_itinerary_result['selected_itinerary'])}")
print()
for day in select_itinerary_result["selected_itinerary"]:
    print(f"\n=== Day {day['day_number']} ===")
    for item in day["itinerary"]:
        p = item["place"]
        print(f"{item['order']}. [{p.get('bucket','?'):8}] {p['name']}")
        print(f"   도착: {item['arrive_at']}  추천이유: {item['recommendation_reason']}")

✅ step: selected
✅ 선택된 동선 day 수: 2


=== Day 1 ===
1. [activity] 제로월드 홍대점
   도착: 09:00  추천이유: 재미있는 방탈출 체험
2. [food    ] 비스트로주라 홍대점
   도착: 11:06  추천이유: 홍대 붐비는 양식 맛집
3. [cafe    ] 카페장쌤
   도착: 12:40  추천이유: 바닐라빈이 매력적인 카페
4. [activity] 윗잔다리어린이공원
   도착: 14:13  추천이유: 윗잔다리 어린이공원 산책
5. [activity] 벌툰 봉화당 홍대입구역점
   도착: 16:15  추천이유: 아늑한 만화카페 벌툰
6. [food    ] 형님 저요
   도착: 18:15  추천이유: 홍대 인기 고기 맛집
7. [lodging ] 니도나도모텔
   도착: 19:50  추천이유: 편안한 니도나도 모텔

=== Day 2 ===
1. [activity] 구름아래소극장
   도착: 09:00  추천이유: 아늑한 소극장
2. [food    ] 이츠모라멘
   도착: 11:05  추천이유: 강릉 라멘 맛집
3. [cafe    ] 토라비
   도착: 12:38  추천이유: 아늑한 홍대 디저트 카페
4. [activity] 케이 탑건 홍대점
   도착: 14:09  추천이유: 케이탑건 홍대 실내 사격
6. [food    ] 하이디라오 홍대지점
   도착: 18:15  추천이유: 웨이팅이 긴 중식 맛집
7. [lodging ] 머큐어 앰배서더 서울 홍대
   도착: 19:49  추천이유: 홍대 근처의 앰배서더 호텔


In [ ]:
# [노드] generate_response
from nodes.generate_response import generate_response

generate_state = {
    **select_itinerary_result,
    "user_input": second_filter_result["user_input"],
}
response_result = generate_response(generate_state)

print(f"✅ step: {response_result['step']}")
print()

import json
print(json.dumps(response_result["response"], ensure_ascii=False, indent=2))

In [ ]:
# LangGraph 그래프 빌드
from langgraph.graph import StateGraph, START, END
from core.state import TravelState

graph_builder = StateGraph(TravelState)

# 노드 등록
graph_builder.add_node("preprocess_input", preprocess_input)
graph_builder.add_node("collect_candidate_pool", collect_candidate_pool)
graph_builder.add_node("first_filter_candidates", first_filter_candidates)
graph_builder.add_node("second_filter_candidates", second_filter_candidates)
graph_builder.add_node("travel_matrix", travel_matrix)
graph_builder.add_node("plan_itinerary", plan_itinerary)
graph_builder.add_node("select_itinerary", select_itinerary)
# graph_builder.add_node("generate_response", generate_response)

# 엣지 (직선 연결)
graph_builder.add_edge(START, "preprocess_input")
graph_builder.add_edge("preprocess_input", "collect_candidate_pool")
graph_builder.add_edge("collect_candidate_pool", "first_filter_candidates")
graph_builder.add_edge("first_filter_candidates", "second_filter_candidates")
graph_builder.add_edge("second_filter_candidates", "travel_matrix")
graph_builder.add_edge("travel_matrix", "plan_itinerary")
graph_builder.add_edge("plan_itinerary", "select_itinerary")
graph_builder.add_edge("select_itinerary", "generate_response")
graph_builder.add_edge("generate_response", END)

graph = graph_builder.compile()

In [ ]:
# 그래프 실행
from core.state import make_initial_state
from core.mocks import mock_user_input

initial_state = make_initial_state(mock_user_input)
state_v1 = await graph.ainvoke(initial_state)

print(f"📍 위치: {state_v1['user_input']['location']}")
print(f"📌 좌표: ({state_v1['user_input']['center_lat']}, {state_v1['user_input']['center_lng']})")
print(f"🔍 Kakao raw 후보: {len(state_v1['candidates'])}개")
print(f"✅ step: {state_v1['step']}")
print(f"✅ next_node: {state_v1.get('next_node', '')}")
print(f"⚠️  warnings: {state_v1['warnings']}")
print(f"❌ errors: {state_v1['errors']}")
print()

print(f"📋 itinerary ({len(state_v1['itinerary'])}개 장소):")
for item in state_v1["itinerary"]:
    p = item["place"]
    print(f"  {item['order']}. [{p.get('bucket','?'):8}] {p['name']}")
    print(f"     도착: {item['arrive_at']}  출발: {item['leave_at']}  다음까지: {item['travel_to_next_minutes']}분")

if state_v1.get("violations"):
    print(f"\n❌ violations:")
    for v in state_v1["violations"]:
        print(f"  - {v['reason']}: {v['detail']}")

In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))